# Kidney Disease Prediction for Beginners

Hi! In this notebook I am trying to predict whether a person has Chronic Kidney Disease (CKD) based on simple questions about their lifestyle and health habits.

I am using a dataset that contains answers from normal people about things like their sleep quality, alcohol consumption, fatigue, and blood pressure.

**Goal:** Predict if someone has kidney disease (1 = CKD, 0 = No CKD) based on these simple questions.

## Importing Libraries

First I will import all the libraries that I need to work with the data and visualize it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import pickle
import os

## 1. Loading the Data

Let me load the dataset from the CSV file and look at the first few rows.

In [ ]:
df = pd.read_csv('Chronic_Kidney_Dsease_data.csv')
print('Shape of data:', df.shape)
print('Missing values:', df.isnull().sum().sum())
df.head(3)

target = 'Diagnosis'

# checking how many people have kidney disease vs don't have
print('Target distribution:')
print(df[target].value_counts())
print()
print('Number of people with CKD:', df[target].sum())
print('Number of people with No CKD:', (df[target] == 0).sum())

print("\nNote: Most people in this dataset have kidney disease, which means the data is very unbalanced!")

## 2. Choosing Important Features

The dataset has many columns, but I only want to keep the ones that are easy for a normal person to answer without doing complex medical tests.

I am picking these features:
- `Age`, `Gender`, `BMI`
- `Smoking`, `AlcoholConsumption`, `PhysicalActivity`
- `DietQuality`, `SleepQuality`
- `FamilyHistoryKidneyDisease`, `FamilyHistoryHypertension`, `FamilyHistoryDiabetes`
- `PreviousAcuteKidneyInjury`, `UrinaryTractInfections`
- `SystolicBP`, `DiastolicBP`
- `Edema`, `FatigueLevels`, `NauseaVomiting`, `MuscleCramps`, `Itching`

In [ ]:
# these are the easy features i am using
simple_features = [
    'Age', 'Gender', 'BMI', 'Smoking', 'AlcoholConsumption',
    'PhysicalActivity', 'DietQuality', 'SleepQuality',
    'FamilyHistoryKidneyDisease', 'FamilyHistoryHypertension', 'FamilyHistoryDiabetes',
    'PreviousAcuteKidneyInjury', 'UrinaryTractInfections',
    'SystolicBP', 'DiastolicBP',
    'Edema', 'FatigueLevels', 'NauseaVomiting', 'MuscleCramps', 'Itching'
]

# Create a working copy of the dataframe
df_work = df[simple_features + [target]].copy()

print('I picked', len(simple_features), 'simple features.')
print()

# Let's see which features are most strongly related to kidney disease
print('Correlation with Kidney Disease:')
corr = df_work[simple_features].corrwith(df_work[target]).abs().sort_values(ascending=False)
print(corr.round(3))

## 3. Exploratory Data Analysis (EDA)

Now I will make some graphs to visually understand the data and see what factors are most common for people with kidney disease.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Kidney Disease - Lifestyle Feature Analysis', fontsize=14, fontweight='bold')

# plot 1 - Class balance
df_work[target].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['mediumseagreen','mediumpurple'], edgecolor='black', rot=0)
axes[0,0].set_xticklabels(['No CKD','CKD'])
axes[0,0].set_title('Class Distribution (very uneven!)')

# plot 2 - Age
df_work[df_work[target]==0]['Age'].hist(bins=20, ax=axes[0,1], alpha=0.7, color='mediumseagreen', label='No CKD')
df_work[df_work[target]==1]['Age'].hist(bins=20, ax=axes[0,1], alpha=0.7, color='mediumpurple', label='CKD')
axes[0,1].set_title('Age Distribution by Outcome')
axes[0,1].legend()

# plot 3 - BMI
df_work[df_work[target]==0]['BMI'].hist(bins=20, ax=axes[0,2], alpha=0.7, color='mediumseagreen', label='No CKD')
df_work[df_work[target]==1]['BMI'].hist(bins=20, ax=axes[0,2], alpha=0.7, color='mediumpurple', label='CKD')
axes[0,2].set_title('BMI Distribution')
axes[0,2].legend()

# plot 4 - Symptoms comparison
symptom_cols = ['FatigueLevels','NauseaVomiting','MuscleCramps','Itching']
ckd_means     = [df_work[df_work[target]==1][c].mean() for c in symptom_cols]
no_ckd_means  = [df_work[df_work[target]==0][c].mean() for c in symptom_cols]
x = np.arange(len(symptom_cols))
axes[1,0].bar(x-0.18, no_ckd_means, 0.35, label='No CKD', color='mediumseagreen', edgecolor='k')
axes[1,0].bar(x+0.18, ckd_means,    0.35, label='CKD',    color='mediumpurple',   edgecolor='k')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(symptom_cols, rotation=15, ha='right')
axes[1,0].set_title('Symptom Severity')
axes[1,0].legend()

# plot 5 - BP comparison
bp_data = [
    df_work[df_work[target]==0]['SystolicBP'].values,
    df_work[df_work[target]==1]['SystolicBP'].values
]
axes[1,1].boxplot(bp_data, labels=['No CKD','CKD'])
axes[1,1].set_title('Systolic Blood Pressure')
axes[1,1].set_ylabel('Systolic BP (mmHg)')

# plot 6 - Correlation chart
corr.sort_values().plot(kind='barh', ax=axes[1,2], color='mediumpurple', edgecolor='k')
axes[1,2].set_title('Feature Correlation with CKD')

plt.tight_layout()
plt.savefig('kidney_eda.png', bbox_inches='tight')
plt.show()
print("EDA plot saved.")

## 4. Handling Imbalance and Splitting Data

Because 92% of the people in this dataset have kidney disease, a model might just learn to always guess "Yes" and ignore everything else.

To fix this, I will lower the amount of "CKD" people so it's a bit closer to the number of healthy people.

Then, I will split the data into a training set (to teach the model) and a testing set (to test the model).

In [ ]:
df_majority = df_work[df_work[target]==1]  # People with CKD
df_minority = df_work[df_work[target]==0]  # People with No CKD

print('Before balancing:')
print('  CKD:', len(df_majority))
print('  No CKD:', len(df_minority))

# Downsample the CKD cases so that it's mostly balanced (at most 3 times the healthy people)
n_samples = min(len(df_majority), len(df_minority) * 3)
df_majority_down = resample(df_majority, replace=False, n_samples=n_samples, random_state=42)

# Combine the downsampled majority and the original minority
df_balanced = pd.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

print()
print('After balancing:')
print(df_balanced[target].value_counts())

X = df_balanced[simple_features]
y = df_balanced[target]

# Split 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# We scale the features so that different ranges (like BMI vs Age) don't confuse the model
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print()
print('Training samples (for learning):', X_train.shape[0])
print('Testing samples (for checking):', X_test.shape[0])

## 5. Training the Model

Now I will train a simple Logistic Regression model using the training data so we can save it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# create and train the model
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_sc, y_train)

# test it
predictions = final_model.predict(X_test_sc)
accuracy = accuracy_score(y_test, predictions)

print(f'Model Accuracy: {accuracy * 100:.2f}%')

## 5. Saving the Model

Finally, I will save the trained model, the scaler, and the list of features so I can use them in a web application.

In [ ]:
os.makedirs('models', exist_ok=True)

pickle.dump(final_model,     open('models/kidney_model.pkl',    'wb'))
pickle.dump(scaler,          open('models/kidney_scaler.pkl',   'wb'))
pickle.dump(simple_features, open('models/kidney_features.pkl', 'wb'))

print('Saved successfully!')
print()
print('Web app inputs for kidney disease will need:')
for f in simple_features:
    print(' -', f)